# Import Library

In [ ]:
!pip install youtube-comment-downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 6.8 MB/s eta 0:00:00


In [ ]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 27.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import time
import random
import yt_dlp
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR

# Data Scrapping

In [ ]:
KEYWORDS = [
    "Cafe Aesthetic Padang",
    "Coffee Shop Hits Bukittinggi",
    "Tempat Nongkrong Viral Sumbar",
    "Hidden Gem Sumatera Barat",
    "Staycation Padang Murah",
    "Review iPhone Bahasa Minang",
    "Unboxing HP Murah Padang",
    "Toko Komputer Padang",
    "Modifikasi Motor Sumbar",
    "Vlog Anak Rantau Minang",
    "Mahasiswa Minang di Luar Negeri",
    "Anak Jaksel vs Anak Padang",
    "Pulang Basamo Lebaran Vlog",
    "Food Vlogger Padang",
    "Review Nasi Kapau Viral",
    "Kuliner Malam Bukittinggi",
    "Lagu Minang Remix TikTok",
    "Cinematic Video Sumbar",
    "Podcast Minang Lucu",
    "Reaction Video Lagu Minang",
    "Mobile Legends Turnamen Padang",
    "PUBG Mobile Player Sumbar",
    "Rental PS5 Padang",
    "Live Streamer Minang",
    "Review Warnet Padang",
    "Valorant Indonesia Minang",
    "Modifikasi Nmax Sumbar",
    "Bus Sumbar Viral",
    "Review Mobil Bekas Padang",
    "Sunmori Bukittinggi",
    "Kopdar Komunitas Motor Padang",
    "Bengkel Modif Padang",
    "Thrifting Padang Pasar Raya",
    "Bukittinggi Second Brand",
    "Outfit Lebaran Minang",
    "Distro Lokal Padang",
    "Unboxing Paket Padang",
    "Sneakers Original Padang",
    "Vlog Wisuda UNAND",
    "Anak Kos Padang",
    "Vlog KKN UNP",
    "Belajar Bahasa Inggris Minang",
    "Beasiswa Luar Negeri Sumbar",
    "Skripsi Mahasiswa Padang",
    "Jalan Rusak Sumatera Barat",
    "Info Sumbar Viral Hari Ini",
    "Klarifikasi Selebgram Padang",
    "Pembangunan Tol Sumbar",
    "Banjir Padang Terkini",
    "Hip Hop Minang Terbaru",
    "Cover Lagu Minang Akustik",
    "Stand Up Comedy Padang",
    "Film Pendek Minang Lucu",
    "DJ Minang Full Bass",
    "Rumah Dijual Padang",
    "Investasi Tanah Sumbar",
    "Bisnis Anak Muda Padang",
    "Lowongan Kerja Padang",
]

In [ ]:
VIDEOS_PER_KEYWORD = 5
COMMENTS_PER_VIDEO = 250
OUTPUT_FILE = "comments.csv"

In [ ]:
def search_yt_videos(keyword, limit=3):
  ydl_otps = {
      "default_search": f"ytsearch{limit}",
      "quiet": True,
      "no_warnings": True,
      "ignoreerrors": True,
      "extract_flat": "in_playlist",
      "skip_download": True,
  }

  found_videos = []

  try:
    with yt_dlp.YoutubeDL(ydl_otps) as ydl:
      result = ydl.extract_info(keyword, download=False)
      if "entries" in result:
        for entry in result["entries"]:
          url = entry.get("url")

          if "/shorts/" in url:
            url = url.replace("/shorts/", "/watch?v=")

          if not url.startswith("http"):
            url = f"https://youtube.com/watch?v={url}"

          found_videos.append({
              "link": url,
              "title": entry.get("title", "No Title")
          })
  except Exception as e:
    print(f"Error Youtube Search: {e}")

  return found_videos

def data_scrapping():
  downloader = YoutubeCommentDownloader()
  all_data = []
  processed_urls = set()

  print("Starting Data Scrapping!")

  start_time = time.time()

  for idx, keyword in enumerate(KEYWORDS):
    try:
      results = search_yt_videos(keyword, limit=VIDEOS_PER_KEYWORD)

      if not results:
        print(f"Videos not found for {keyword}")
        continue

      for video in results:
        url = video["link"]
        title = video["title"]

        if url in processed_urls:
          continue
        processed_urls.add(url)

        print(f"Scrapping {title[:40]}...")

        try:
          comments = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)
          count = 0

          for comment in comments:
            text = comment["text"]

            if len(text.split()) > 2:
              all_data.append({
                  "keyword": keyword,
                  "video_title": title,
                  "text": text,
                  "url": url,
              })

              count += 1
            if count >= COMMENTS_PER_VIDEO:
              break

          print(f"Found {count} comments for {title}")

          time.sleep(random.uniform(5,10))

        except Exception as e:
          print(f"Failed to get comments: {e}")

    except Exception as e:
      print(f"Failed to search video: {e}")

    if (idx + 1) % 5 == 0:
      pd.DataFrame(all_data).to_csv(OUTPUT_FILE, index=False, encoding="utf-8", sep='|')
      print(f"Auto Save Comments: {len(all_data)} so far")

  df = pd.DataFrame(all_data)
  df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8", sep=';')

  print("Data Scrapping Finished!")


In [ ]:
data_scrapping()

Starting Data Scrapping!
Scrapping Cafe/baked house aesthetic di Bukittingg...
Found 0 comments for Cafe/baked house aesthetic di Bukittinggi ,Croissants dan crofle 🥐🧇
Scrapping The Best Cafés to Work From in Uluwatu, ...
Found 17 comments for The Best Cafés to Work From in Uluwatu, Bali - 2022
Scrapping [Vlog] one fine day : strolling around p...
Found 10 comments for [Vlog] one fine day : strolling around padang city ,aesthetic place, cafes, etc.
Scrapping Coffee Shop design - Coffe Shop Pertama ...
Found 0 comments for Coffee Shop design - Coffe Shop Pertama di Kota Padang  dengan konsep entertainment
Scrapping Palano Hill Cafe, Hidden gem cafe aesthe...
Found 4 comments for Palano Hill Cafe, Hidden gem cafe aesthetic terbaru di kota Payakumbuh, Sumatera Barat. #cafehits
Scrapping VOCA KOPI ||| CAFE BUKITTINGGI...
Found 0 comments for VOCA KOPI ||| CAFE BUKITTINGGI
Scrapping tristant resto and coffee shop  ,BUKITTI...
Found 0 comments for tristant resto and coffee shop  ,BUKITTINGGI